# **E-Commerce Customer Analytics**
### **Data Cleaning**

**Dataset:** UCI Online Retail Dataset  
**Notebook:** 02 of 06  
**Goal:** Remove everything that would corrupt our analysis. Every decision made here was justified by what we found in `01_eda.ipynb`. We are not making arbitrary choices - we are fixing known, documented problems.

**Import Libraries and Setup**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")

os.makedirs("/content/data/processed", exist_ok=True)

**Load Raw Data**

We load the original file exactly as it came. Nothing has been touched yet.

In [2]:
df = pd.read_excel("/content/Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


**Drop Rows With Missing CustomerID**

As established in EDA, **135,080 rows have no CustomerID.** Without a CustomerID, a transaction cannot be linked to any customer. It is invisible to every model we build.

There is no way to impute or recover these rows. So we let them go.

In [3]:
before = len(df)

df = df.dropna(subset=["CustomerID"])

In [4]:
after = len(df)
dropped = before - after

In [5]:
print(f"Rows before : {before}")
print(f"Rows after : {after}")
print(f"Rows dropped : {dropped}")

Rows before : 541909
Rows after : 406829
Rows dropped : 135080


**Remove Cancelled Orders**

Cancelled orders have invoice numbers that start with the letter `C`. These are not purchases, they are reversals. Including them would undercount revenue, distort frequency metrics, and corrupt LTV calculations.

In [6]:
before = len(df)

cancelled = df["InvoiceNo"].astype(str).str.startswith("C")
df = df[~cancelled]

In [7]:
after = len(df)
dropped = before - after

print(f"Rows before : {before}")
print(f"Rows after : {after}")
print(f"Cancelled orders removed : {dropped}")

Rows before : 406829
Rows after : 397924
Cancelled orders removed : 8905


In [8]:
# Remove Negative and Zero Quantities
before = len(df)

df = df[df["Quantity"] > 0]

In [9]:
after = len(df)
dropped = before - after

print(f"Rows before : {before}")
print(f"Rows after : {after}")
print(f"Rows dropped : {dropped}")

Rows before : 397924
Rows after : 397924
Rows dropped : 0


**Filter to United Kingdom Only**

The dataset spans 38 countries, but **91.4% of all transactions are from the UK.** Keeping all countries would create inconsistency, different pricing, buying habits, and order sizes across markets and would distort any customer-level model we build.

We filter to UK only. This is the market we are modelling.

In [10]:
before = len(df)

df = df[df["Country"] == "United Kingdom"]

In [11]:
after = len(df)
dropped = before - after

print(f"Rows before : {before}")
print(f"Rows after  : {after}")
print(f"Non-UK rows removed: {dropped}")

Rows before : 397924
Rows after  : 354345
Non-UK rows removed: 43579


In [12]:
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df.dtypes

,0
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
UnitPrice,float64
CustomerID,object
Country,object


In [13]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [14]:
# Total Price
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
print(df[["Quantity", "UnitPrice", "TotalPrice"]].head(5))

   Quantity  UnitPrice  TotalPrice
0         6       2.55       15.30
1         6       3.39       20.34
2         8       2.75       22.00
3         6       3.39       20.34
4         6       3.39       20.34


In [15]:
# Remove Duplicate

before = len(df)

df = df.drop_duplicates()

In [16]:
after = len(df)
dropped = before - after

print(f"Rows before : {before}")
print(f"Rows after  : {after}")
print(f"Duplicates removed: {dropped}")

Rows before : 354345
Rows after  : 349227
Duplicates removed: 5118


In [17]:
# Sanity Check
print("Final Dataset Shape")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

print("\nMissing Values")
print(df.isnull().sum())

print("\nData Types")
print(df.dtypes)

print("\nTotalPrice Summary")
print(df["TotalPrice"].describe())

Final Dataset Shape
Rows    : 349227
Columns : 9

Missing Values
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

Data Types
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
TotalPrice            float64
dtype: object

TotalPrice Summary
count    349227.000000
mean         20.860428
std         328.406035
min           0.000000
25%           4.200000
50%          10.200000
75%          17.850000
max      168469.600000
Name: TotalPrice, dtype: float64


In [18]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [19]:
# Save Cleaned Data
output_path = "/content/data/processed/online_retail_cleaned.csv"
df.to_csv(output_path, index=False)